# 01 Load .mat and build inventory
Load MACHINE_Data.mat, RFM_DATA.mat, OES_DATA.mat; inspect keys/structures; build inventory (wafer_id, exp_id, label, lengths, variable list). Output: data/interim/inventory.parquet

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(".").resolve()
if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))
from app.etchfdc.io.mat_reader import load_mat, inspect_mat

DATA_RAW = ROOT / "data" / "raw"
DATA_INTERIM = ROOT / "data" / "interim"
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

In [ ]:
mat_files = ["MACHINE_Data.mat", "RFM_DATA.mat", "OES_DATA.mat"]
inventory_rows = []
for fname in mat_files:
    path = DATA_RAW / fname
    if not path.exists():
        print(f"Skip (not found): {path}")
        continue
    data = load_mat(path)
    info = inspect_mat(path)
    for key in data:
        v = data[key]
        try:
            length = len(v) if hasattr(v, "__len__") else getattr(v, "shape", (None,))[0]
        except Exception:
            length = None
        inventory_rows.append({
            "file": fname,
            "variable": key,
            "length": length,
            "keys_inspected": list(info.get(key, {}))
        })
inv = pd.DataFrame(inventory_rows)
inv.to_parquet(DATA_INTERIM / "inventory.parquet", index=False)
print(inv)